<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">Lab 04: トラベルエージェントをトレースする</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Microsoft Foundry エージェント可観測性
  </p>
</div>

## 学習内容

このラボでは、Contoso Travel エージェントに **OpenTelemetry トレース** を組み込みます。まずローカルデバッグ用に **コンソール** へ出力し、次に本番環境の可観測性のために **Azure Monitor** へ接続します。エージェント内部で何が起きているか、ユーザークエリ、ツール呼び出し、モデル呼び出し、レスポンス生成まで、すべてを正確に確認できます。

---

## パート A: コンソールトレース

## 2. セットアップ

---


In [ ]:
# 環境変数を読み込み、GenAI トレースフラグを有効化する
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

# トレースモジュールをインポートする前に設定する必要があります
# Azure SDK の実験的な GenAI スパン生成を有効化します
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
# トレースにプロンプト/レスポンスの全コンテンツをキャプチャします
# コンテンツに PII が含まれる場合は本番環境で無効化してください
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"

print("✅ GenAI トレース用の環境変数が設定されました")


In [ ]:
# Azure SDK 呼び出しをトレースするように OpenTelemetry を設定する
from azure.core.settings import settings
# Azure SDK が HTTP 呼び出しの OTel スパンを出力するよう指示する
settings.tracing_implementation = "opentelemetry"

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from azure.ai.projects.telemetry import AIProjectInstrumentor

# ConsoleSpanExporter はスパンを stdout に出力します（開発専用）
span_exporter = ConsoleSpanExporter()
# TracerProvider はスパンの作成とライフサイクルを管理します
tracer_provider = TracerProvider()
# SimpleSpanProcessor は各スパンを同期的にエクスポートします；
# 本番環境ではパフォーマンスのために BatchSpanProcessor を使用してください
tracer_provider.add_span_processor(SimpleSpanProcessor(span_exporter))
trace.set_tracer_provider(tracer_provider)
tracer = trace.get_tracer("contoso-travel-tracing")

# Azure AI SDK を自動インストルメント化：エージェント呼び出し、ツール
# 呼び出し、モデルリクエストが自動的にスパンを出力します
AIProjectInstrumentor().instrument()

print("✅ コンソールトレースが設定されました")
print("   トレースは各セルの下の出力に表示されます")

In [ ]:
# Microsoft Foundry に接続（ここでは allow_preview は不要
# — トレースは GA フィーチャーで動作します）
from azure.identity import DefaultAzureCredential, AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

tenant_id = os.getenv("TENANT_ID")
endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = AzureCliCredential(tenant_id=tenant_id)
#credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Microsoft Foundry に接続しました")

## 3. トラベルエージェントの作成とトレース

---


In [ ]:
# 可観測性デモ用のシンプルなトレース対応エージェントを作成する
agent = project_client.agents.create_version(
    agent_name="contoso-travel-traced",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="あなたは Contoso Travel のコンシェルジュです。目的地、フライト、ホテル、レンタカーに関する旅行の質問をサポートしてください。",
    ),
)
print(f"✅ エージェントを作成しました: {agent.name} (v{agent.version})")

## 4. トレース付きトラベルクエリの実行

このセルを実行して、下のコンソール出力を確認してください — 各操作に対して OpenTelemetry スパンが表示されます。

---


In [ ]:
# start_as_current_span はペアレントスパンを作成します；SDK の
# 内部呼び出しはすべて同じトレース内の子スパンになります
with tracer.start_as_current_span("contoso-travel-query"):
    conversation = openai_client.conversations.create()

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="ヨーロッパの夏休みに最適な旅行先はどこですか？",
    )

    print(f"\n🤖 エージェントのレスポンス:\n{response.output_text}")
# 上のコンソール出力で trace_id、span_id、
# parent_id、トークン数などの属性を確認してください

## 5. トレース出力の解釈

上のコンソール出力には、次のようなトレーススパンが含まれています：

- **`contoso-travel-query`** — 作成したペアレントスパン
  - **`create_conversation`** — 会話の作成
  - **`invoke_agent`** — モデル推論呼び出し

各スパンには以下が含まれます：
- `trace_id` — 1つのトレース内のすべてのスパンをグループ化
- `span_id` — このスパンの一意なID
- `parent_id` — 子スパンを親にリンク
- `attributes` — モデル名やトークン数などのメタデータ

---


## パート B: Azure Monitor トレース

## 6. Azure Monitor トレースの設定

---


In [ ]:
# 重複スパンを避けるためにコンソールトレーサーをシャットダウンする
tracer_provider.shutdown()

from azure.monitor.opentelemetry import configure_azure_monitor

# このプロジェクトにリンクされた App Insights 接続文字列を取得します
# — 手動設定は不要です
app_insights_connection_string = project_client.telemetry.get_application_insights_connection_string()

# コンソールエクスポーターを Azure Monitor エクスポーターに置き換えます
# 新しい TracerProvider + BatchSpanProcessor を設定して
# スパンを Application Insights に自動送信します
configure_azure_monitor(connection_string=app_insights_connection_string)

# プロバイダーを再設定した後は新しいトレーサーを取得する必要があります
tracer = trace.get_tracer("contoso-travel-tracing")

print("✅ Azure Monitor トレースが設定されました")
print(f"   トレースは Application Insights に送信されます")
print(f"   Foundry ポータル → Tracing タブで確認できます")

## 7. トレース付きトラベルクエリの実行（Azure Monitor）

このトレースは Foundry ポータルに送信されます。トレースが Azure Monitor に表示されるまで数分かかる場合があります。

---


In [ ]:
# Azure Monitor エクスポーターを使用してトレース付きクエリを実行する
with tracer.start_as_current_span("contoso-travel-paris-query") as span:
    # カスタム属性により Azure Monitor でのフィルタリングが可能になります
    # 一貫した名前空間（例："travel."、"contoso."）を使用して
    # 目的地やエージェントでトレースをクエリできます
    span.set_attribute("travel.destination", "Paris")
    span.set_attribute("travel.query_type", "trip_planning")
    span.set_attribute("contoso.agent_name", agent.name)

    conversation2 = openai_client.conversations.create()

    response = openai_client.responses.create(
        conversation=conversation2.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="パリへの1週間の旅行を計画したいです。費用、天気、必見スポットについて教えてください。",
    )

    print(f"🤖 エージェントのレスポンス:\n{response.output_text}")
    print(f"\n📊 トレースが Azure Monitor に送信されました！")
    # 16進数の trace_id — ポータルでトレースを検索するのに使用します
    print(f"   トレース ID: {span.get_span_context().trace_id:032x}")

## 8. Foundry ポータルでトレースを確認する

トレースを確認するには：

1. [Microsoft Foundry ポータル](https://ai.azure.com) にアクセス
2. プロジェクトを開く
3. エージェントの **Tracing** タブをクリック
4. 定義したスパン名でトレースの一覧が表示されます
5. トレースをクリックしてスパンツリー全体を確認


注意:トレースが実行後に Azure Monitor に表示されるまで2〜5分かかる場合があります。


---


## 9. カスタムスパン属性

カスタム属性はフィルタリングと分析においてトレースをより有用にします。ビジネスレベルのクエリを可能にするために、旅行固有のメタデータを追加します。

---


In [ ]:
# ビジネスレベルのフィルタリングのためのより詳細なカスタム属性
# これらは Azure Monitor の customDimensions に表示され、
# App Insights で KQL を使用してクエリできます
with tracer.start_as_current_span("contoso-travel-london-query") as span:
    span.set_attribute("travel.destination", "London")
    span.set_attribute("travel.query_type", "budget_inquiry")
    span.set_attribute("travel.customer_segment", "business")
    span.set_attribute("contoso.agent_name", agent.name)
    span.set_attribute("contoso.agent_version", str(agent.version))

    response = openai_client.responses.create(
        conversation=conversation2.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="ニューヨークからロンドンへのビジネスクラスの旅行オプションを教えてください。",
    )

    print(f"🤖 エージェントのレスポンス:\n{response.output_text}")
    print(f"\n📊 トレースにカスタム属性が追加されました：")
    print(f"   travel.destination = London")
    print(f"   travel.query_type = budget_inquiry")
    print(f"   travel.customer_segment = business")

## 10. クリーンアップ

---


In [ ]:
# クリーンアップ: リモートリソースを削除してクライアントを閉じる
openai_client.conversations.delete(conversation_id=conversation.id)
openai_client.conversations.delete(conversation_id=conversation2.id)
print("✅ 会話を削除しました")

project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("✅ エージェントを削除しました")

openai_client.close()
project_client.close()
credential.close()
print("✅ クライアントを閉じました")